# Test de Acceso a Datos y Conteo de Tablas

**Objetivo:** Este cuaderno sirve para verificar la conexión a la base de datos `employees` y para contar el número de registros en cada una de las tablas.

In [7]:
import pandas as pd
import psycopg2
import getpass

### 1. Conexión a la Base de Datos
Para conectarte, la base de datos debe estar corriendo en el contenedor de Docker. Introduce las credenciales cuando se te soliciten.

Las credenciales por defecto (si no las has cambiado en tu entorno) suelen ser:
- **Database Name:** `employees`
- **Database User:** `admin`
- **Database Password:** `admin`

In [9]:
db_name = input("Database Name: ")
db_user = input("Database User: ")
db_pass = getpass.getpass("Database Password: ")
db_host = "postgres"  # El nombre del servicio en docker-compose
db_port = "5432"

try:
    conn = psycopg2.connect(
        dbname=db_name,
        user=db_user,
        password=db_pass,
        host=db_host,
        port=db_port
    )
    print("✅ Conexión a la base de datos exitosa.")
    cur = conn.cursor()
except psycopg2.OperationalError as e:
    print(f"❌ Error en la conexión: {e}")
    conn = None

Database Name:  employees
Database User:  admin
Database Password:  ········


✅ Conexión a la base de datos exitosa.


### 2. Conteo de Registros por Tabla

In [11]:
if conn:
    try:
        # Obtener la lista de tablas del esquema público
        cur.execute("""
            SELECT tablename 
            FROM pg_catalog.pg_tables 
            WHERE schemaname = 'public';
        """)
        tables = [row[0] for row in cur.fetchall()]
        
        counts = {}
        print("Contando registros en cada tabla...")
        for table in tables:
            count_query = f'SELECT COUNT(*) FROM "{table}";'
            cur.execute(count_query)
            count = cur.fetchone()[0]
            counts[table] = count
            
        # Imprimir los resultados en una tabla de Markdown
        print("| Tabla                  | Número de Registros |")
        print("|------------------------|---------------------|")
        for table, count in counts.items():
            print(f"| {table:<22} | {count:<19} |")

    except Exception as e:
        print(f"Ocurrió un error durante el conteo: {e}")
    finally:
        # Cerrar el cursor y la conexión
        if 'cur' in locals() and cur:
            cur.close()
        conn.close()
        print("Conexión cerrada.")
else:
    print("No hay conexión a la base de datos para realizar el conteo.")

Contando registros en cada tabla...
| Tabla                  | Número de Registros |
|------------------------|---------------------|
| salaries               | 2844047             |
| employee_payment_history | 1                   |
| dept_manager           | 24                  |
| departments            | 9                   |
| dept_emp               | 331603              |
| titles                 | 443308              |
| employees              | 300024              |
Conexión cerrada.
